# Simplifying Regular Expressions with Flattened Sums and Products

A simple way to simplify regular expressions is to rewrite them with rules like
$R + \emptyset \rightarrow R$.  Such rules work on the syntax tree of a regular expression, where a sum
$r_1 + r_2 + r_3$ is either the tree $(r_1 + r_2) + r_3$ or the tree $r_1 + (r_2 + r_3)$.  A rule matches only
one of these trees.  Furthermore, two summands of a sum are only compared if they happen to be neighbours in
the tree.  Therefore, the sum $\texttt{b} + \texttt{a} + \texttt{b}^*$ is not simplified to $\texttt{a} + \texttt{b}^*$,
although $L(\texttt{b}) \subseteq L(\texttt{b}^*)$.

In this notebook we represent a sum as the *set* of its summands and a product as the *list* of its factors.
Then the laws of associativity, commutativity, and idempotence of $+$, i.e.
$$ (r_1 + r_2) + r_3 \doteq r_1 + (r_2 + r_3), \quad r_1 + r_2 \doteq r_2 + r_1, \quad\mbox{and}\quad r + r \doteq r, $$
and the law of associativity of $\cdot$ are built into the representation.  The other laws from Chapter 2 of
the lecture notes are applied by *smart constructors*:  These are functions that build a sum, a product, or a
Kleene closure and simplify it at the same time.  Since the smart constructors are applied bottom-up, every
subexpression is already simplified when it becomes part of a larger expression.  Therefore, a single pass over
the regular expression suffices.

In [ ]:
#use "../lib/nb.ml";;
run_notebook "Types.ipynb"

Several functions of this notebook are called many times with the same arguments.  The Python version uses the
decorator `@cache` from the module `functools`, which stores the results of a function in a dictionary.  OCaml has
no decorators, but we can write the function `cache` ourselves: `cache f` returns a function that stores every
result in a *hash table* and returns the stored result when it is called again with the same argument.  Since the
cached functions are recursive, their recursive calls have to go through the cache, too.  Therefore, `f` gets the
cached function as its first argument.

The standard hash function of OCaml only looks at the first few nodes of a tree.  Since the regular expressions
can become very large, the key of the hash table is the pair consisting of a hash value that looks at up to 100
nodes and the argument itself.

In [ ]:
let cache (f : ('a -> 'b) -> 'a -> 'b) : 'a -> 'b =
  let table = Hashtbl.create 1024 in
  let rec self x =
    let key = (Hashtbl.hash_param 100 1000 x, x) in
    match Hashtbl.find_opt table key with
    | Some y -> y
    | None ->
        let y = f self x in
        Hashtbl.replace table key y;
        y
  in
  self

The type `regexp` describes regular expressions as they are used in the other notebooks.  It is defined in the
notebook `Types.ipynb`.

In [ ]:
#show_type regexp

The type `flat` describes the *flattened* representation of regular expressions:
- A character `c` is represented as `FChar c`.
- The Kleene closure $r^*$ is represented as `FStar r`.
- The sum $r_1 + \cdots + r_n$ is represented as `FSum s`, where `s` is the set $\{r_1, \cdots, r_n\}$.  None of the
  summands $r_i$ is itself a sum.
- The product $r_1 \cdot \ldots \cdot r_n$ is represented as `FProd [r1; ...; rn]`.  None of the
  factors $r_i$ is itself a product.

The empty regular expression $\emptyset$ is the empty sum and $\varepsilon$ is the empty product, since $\emptyset$
is the neutral element of $+$ and $\varepsilon$ is the neutral element of $\cdot$.  These two are called `empty` and
`epsilon`.  Apart from these two, every sum has at least two summands and every product has at least two factors.

Since every set of type `FSet.t` has exactly one representation, two flattened regular expressions are equal iff
they are equal according to the operator `=`, and they can be compared with `compare`.

In [ ]:
type flat =
  | FChar of char           (* a character c                                  *)
  | FStar of flat           (* r*                                             *)
  | FSum  of flat FSet.t    (* r1 + ... + rn, none of the summands is a sum   *)
  | FProd of flat list      (* r1 ⋅ ... ⋅ rn, none of the factors is a product *)

let empty   : flat = FSum FSet.empty    (* ∅ is the empty sum     *)
let epsilon : flat = FProd []           (* 𝜀 is the empty product *)

## Pretty Printing and Size

The functions `precedence`, `arg_str`, and `to_str` are adapted from the notebook `Chapter-02/Regular-Expressions.ipynb`.
The function `to_str r` turns the regular expression `r` into a string that only contains the parentheses
that are needed.

**Function `precedence r`**
- *Input:* `r` is a regular expression.
- *Output:* The precedence of the operator at the root of `r`.  Atoms have the precedence $4$.

In [ ]:
let precedence (r : regexp) : int =
  match r with
  | Empty | Epsilon | Char _ -> 4
  | Star _                   -> 3
  | Product _                -> 2
  | Sum _                    -> 1

**Function `arg_str(r, p)`**
- *Input:* `r` is a regular expression and `p` is a precedence.
- *Output:* The string representing `r`, enclosed in parentheses if the precedence of `r` is lower than `p`.

**Function `to_str r`**
- *Input:* `r` is a regular expression.
- *Output:* The string representing `r`, using as few parentheses as possible.

The functions `arg_str` and `to_str` call each other.  Therefore, they are defined together in the next cell
using `let rec ... and ...`.

In [ ]:
let rec arg_str (r : regexp) (p : int) : string =
  if precedence r < p then "(" ^ to_str r ^ ")"
  else to_str r

and to_str (r : regexp) : string =
  match r with
  | Empty   -> "∅"
  | Epsilon -> "𝜀"
  | Char c  -> String.make 1 c
  | Star r1 -> arg_str r1 3 ^ "*"
  | Sum (r1, r2) | Product (r1, r2) ->
      let p   = precedence r in
      let sep = match r with Sum _ -> " + " | _ -> "⋅" in
      arg_str r1 p ^ sep ^ arg_str r2 (p + 1)

To measure the size of a regular expression we count the nodes of its syntax tree:  Every character, every
occurrence of $\varepsilon$ and $\emptyset$, and every operator counts as one node.

**Function `size r`**
- *Input:* `r` is a regular expression, represented as a value of type `regexp`.
- *Output:* The number of nodes of the syntax tree of `r`.

In [ ]:
let rec size (r : regexp) : int =
  match r with
  | Star r1 -> 1 + size r1
  | Sum (r1, r2) | Product (r1, r2) -> 1 + size r1 + size r2
  | Empty | Epsilon | Char _ -> 1

The function `flat_size r` computes the size of a flattened regular expression `r`.  This is the size of the
syntax tree that `r` represents:  A sum of $n$ summands needs $n - 1$ operators `+` and a product of $n$ factors
needs $n - 1$ operators `⋅`.

**Function `flat_size r`**
- *Input:* `r` is a flattened regular expression.
- *Output:* The number of nodes of the syntax tree that `r` represents.

In [ ]:
let flat_size : flat -> int =
  cache (fun flat_size r ->
    match r with
    | FStar s -> 1 + flat_size s
    | FSum s when not (FSet.is_empty s) -> FSet.fold (fun x n -> n + flat_size x) s 0 + FSet.cardinal s - 1
    | FProd f when f <> [] -> List.fold_left (fun n x -> n + flat_size x) 0 f + List.length f - 1
    | _ -> 1)

## Auxiliary Functions

The function `summands r` returns the set of summands of `r`.  If `r` is not a sum, `r` is its only summand.
Similarly, the function `factors r` returns the list of factors of `r`.

**Function `summands r`**
- *Input:* `r` is a flattened regular expression.
- *Output:* The set of summands of `r`.

In [ ]:
let summands (r : flat) : flat FSet.t =
  match r with
  | FSum s -> s
  | _ -> FSet.singleton r

**Function `factors r`**
- *Input:* `r` is a flattened regular expression.
- *Output:* The list of factors of `r`.

In [ ]:
let factors (r : flat) : flat list =
  match r with
  | FProd f -> f
  | _ -> [ r ]

A regular expression $r$ is *nullable* iff $\varepsilon \in L(r)$.  A Kleene closure is always nullable, a sum
is nullable iff one of its summands is nullable, and a product is nullable iff all of its factors are nullable.
In particular, $\varepsilon$ is nullable as the empty product, while $\emptyset$ is not nullable as the empty sum.

**Function `nullable r`**
- *Input:* `r` is a flattened regular expression.
- *Output:* `true` if $\varepsilon \in L(r)$, `false` otherwise.

In [ ]:
let nullable : flat -> bool =
  cache (fun nullable r ->
    match r with
    | FStar _ -> true
    | FSum s  -> FSet.exists nullable s
    | FProd f -> List.for_all nullable f
    | FChar _ -> false)

The function `included r1 r2` checks whether $L(r_1) \subseteq L(r_2)$.  It only inspects the syntax of
$r_1$ and $r_2$.  If it returns `true`, then $L(r_1) \subseteq L(r_2)$ holds.  If it returns `false`, the inclusion
might still hold.  The function uses the following facts:
1. $L(r) \subseteq L(r)$ and $L(\emptyset) \subseteq L(r)$.
2. $L(\varepsilon) \subseteq L(r)$ iff $r$ is nullable.
3. $L(s_1 + \cdots + s_n) \subseteq L(r)$ iff $L(s_i) \subseteq L(r)$ for all $i$.
4. If $L(r) \subseteq L(s_i)$ for some $i$, then $L(r) \subseteq L(s_1 + \cdots + s_n)$.
5. If $L(r) \subseteq L(s)$, then $L(r) \subseteq L(s^*)$.  If $L(t) \subseteq L(s^*)$, then also
   $L(t^*) \subseteq L(s^*)$.  If $L(f_i) \subseteq L(s^*)$ for all $i$, then
   $L(f_1 \cdot \ldots \cdot f_n) \subseteq L(s^*)$, since $L(s^*)$ is closed under concatenation.

**Function `included r1 r2`**
- *Input:* `r1` and `r2` are flattened regular expressions.
- *Output:* `true` if the syntax of `r1` and `r2` shows that $L(r_1) \subseteq L(r_2)$, `false` otherwise.

Since `cache` works with functions of one argument, the cached function `included_cached` takes the pair `(r1, r2)`.

In [ ]:
let included_cached : flat * flat -> bool =
  cache (fun included (r1, r2) ->
    if r1 = r2 || r1 = empty then true
    else if r1 = epsilon then nullable r2
    else
      match r1, r2 with
      | FSum s1, _ -> FSet.for_all (fun s -> included (s, r2)) s1
      | _, FSum s2 -> FSet.exists (fun s -> included (r1, s)) s2
      | _, FStar s2 ->
          included (r1, s2)
          || (match r1 with
              | FStar s1 -> included (s1, r2)
              | FProd f1 -> List.for_all (fun f -> included (f, r2)) f1
              | _ -> false)
      | _ -> false)

let included (r1 : flat) (r2 : flat) : bool = included_cached (r1, r2)

Since sets are unordered, the smart constructors sort summands with the function `sort_key` whenever the
result depends on the order in which they are processed.  In Python, this makes the results independent of the
order in which Python stores the elements of a set.  (In OCaml, the elements of an `FSet.t` are sorted anyway,
but we sort by `sort_key` in order to get the same results as the Python version.)  The key sorts by size first
and then by the string representation.  The function `sorted_by_key` sorts a list by these keys.

The function `sort_key` needs the function `unflatten`, which is defined at the end of this notebook, and the
function `factor_out` needs the smart constructors `mk_sum` and `mk_prod`, which are defined later, too.
Therefore, the next cell creates *forward references* for these functions.  A reference is a mutable box:
`!unflatten_ref` is the function that is currently stored in the box `unflatten_ref`, and once `unflatten` has been
defined, it is stored in its box via the operator `:=`.

In [ ]:
let unflatten_ref : (flat -> regexp) ref       = ref not_yet_defined
let mk_sum_ref    : (flat FSet.t -> flat) ref  = ref not_yet_defined
let mk_prod_ref   : (flat list -> flat) ref    = ref not_yet_defined

**Function `sort_key r`**
- *Input:* `r` is a flattened regular expression.
- *Output:* The pair consisting of the size of `r` and the string representing `r`.

In [ ]:
let sort_key : flat -> int * string =
  cache (fun _ r -> (flat_size r, to_str (!unflatten_ref r)))

let sorted_by_key (l : flat list) : flat list =
  List.stable_sort (fun x y -> compare (sort_key x) (sort_key y)) l

## Smart Constructors for Sums

If $L(x) \subseteq L(y)$ holds for two summands $x$ and $y$ of a sum, then $x + y \doteq y$ and the summand $x$
can be dropped.  For example, $\texttt{b} + \texttt{b}^* \doteq \texttt{b}^*$ and $\varepsilon + r^* \doteq r^*$.
If both $L(x) \subseteq L(y)$ and $L(y) \subseteq L(x)$, only the summand with the smaller key is kept.

**Function `absorb s`**
- *Input:* `s` is a set of summands.
- *Output:* The set of those summands of `s` that are not contained in another summand.

In [ ]:
let absorb (s : flat FSet.t) : flat FSet.t =
  FSet.filter
    (fun x ->
      not (FSet.exists (fun y -> y <> x && included x y && (not (included y x) || sort_key y < sort_key x)) s))
    s

The law $\varepsilon + r^* \cdot r \doteq r^*$ from Chapter 2 and its mirror image
$\varepsilon + r \cdot r^* \doteq r^*$ replace the summands $\varepsilon$ and $r \cdot r^*$ by $r^*$.

**Function `plus_star s`**
- *Input:* `s` is a set of summands.
- *Output:* If `s` contains $\varepsilon$ and a product of the form $r \cdot r^*$ or $r^* \cdot r$, these two
  summands are replaced by $r^*$.  Otherwise, `s` is returned unchanged.

In [ ]:
let plus_star (s : flat FSet.t) : flat FSet.t =
  if not (FSet.mem epsilon s) then s
  else
    let rec loop (xs : flat list) : flat FSet.t =
      match xs with
      | [] -> s
      | x :: rest ->
          let candidate =
            match factors x with
            | [ f; FStar g ] -> Some (f, g)
            | [ FStar g; f ] -> Some (f, g)
            | _ -> None
          in
          match candidate with
          | Some (f, g) when f = g -> FSet.union (FSet.diff s (FSet.of_list [ epsilon; x ])) (FSet.singleton (FStar f))
          | _ -> loop rest
    in
    loop (sorted_by_key (FSet.to_list s))

The laws of distributivity
$$ s \cdot t_1 + s \cdot t_2 \doteq s \cdot (t_1 + t_2) \quad\mbox{and}\quad
   t_1 \cdot s + t_2 \cdot s \doteq (t_1 + t_2) \cdot s $$
are used to factor out common prefixes and suffixes.  A summand $s$ that is not a product counts as the product
$s \cdot \varepsilon$, e.g. $\texttt{a} + \texttt{a} \cdot \texttt{b}^*$ is turned into
$\texttt{a} \cdot (\varepsilon + \texttt{b}^*) \doteq \texttt{a} \cdot \texttt{b}^*$.  Factoring out does not always make
a regular expression smaller.  Therefore, the function `factor_out` tries every first factor and every last factor
that is shared by several summands and returns the smallest of the resulting sums.

**Function `factor_out s`**
- *Input:* `s` is a set of at least two summands.
- *Output:* A flattened regular expression that is equivalent to the sum of `s` and is as small as possible
  after factoring out one common first or last factor.

The summands are grouped by their first factor (`pos = 0`) or by their last factor (`pos = -1`).  The groups are
stored in a dictionary that maps a factor to the list of summands sharing it.  As in the Python version, the
variable `best` is changed in a loop, so it is a reference.

In [ ]:
let factor_out (s : flat FSet.t) : flat =
  let mk_sum = !mk_sum_ref and mk_prod = !mk_prod_ref in
  let last l = List.nth l (List.length l - 1) in
  let all_but_last l = List.filteri (fun i _ -> i < List.length l - 1) l in
  let best = ref (FSum s) in
  List.iter
    (fun pos ->
      let groups =
        List.fold_left
          (fun groups x ->
            if x = epsilon then groups
            else
              let k = if pos = 0 then List.hd (factors x) else last (factors x) in
              Dict.add k (Dict.get k ~default:[] groups @ [ x ]) groups)
          Dict.empty (sorted_by_key (FSet.to_list s))
      in
      List.iter
        (fun f ->
          let xs = Dict.find f groups in
          if List.length xs >= 2 then begin
            let new_ =
              if pos = 0 then
                let rests = FSet.of_list (List.map (fun x -> mk_prod (List.tl (factors x))) xs) in
                mk_prod [ f; mk_sum rests ]
              else
                let rests = FSet.of_list (List.map (fun x -> mk_prod (all_but_last (factors x))) xs) in
                mk_prod [ mk_sum rests; f ]
            in
            let candidate = mk_sum (FSet.add new_ (FSet.diff s (FSet.of_list xs))) in
            if flat_size candidate < flat_size !best then best := candidate
          end)
        (sorted_by_key (Dict.keys groups)))
    [ 0; -1 ];
  !best

The smart constructor `mk_sum s` builds the sum of the regular expressions in the set `s`:
1. Summands that are sums themselves are replaced by their summands.  As $\emptyset$ is the empty sum, it vanishes.
2. Summands that are contained in other summands are dropped.
3. The law $\varepsilon + r \cdot r^* \doteq r^*$ is applied.
4. A sum with only one summand is this summand and a sum without summands is $\emptyset$.
5. Common prefixes and suffixes are factored out.

**Function `mk_sum s`**
- *Input:* `s` is a set of flattened regular expressions.
- *Output:* A simplified flattened regular expression that is equivalent to the sum of the elements of `s`.

In [ ]:
let mk_sum : flat FSet.t -> flat =
  cache (fun _ s ->
    let s = FSet.(let* x = s in summands x) in
    let s = plus_star (absorb s) in
    match FSet.to_list s with
    | [] -> empty
    | [ r ] -> r
    | _ -> factor_out s)
;;
mk_sum_ref := mk_sum

## Smart Constructors for Products and Kleene Closures

The smart constructor `mk_prod f` builds the product of the regular expressions in the list `f`:
1. If one of the factors is $\emptyset$, the product is $\emptyset$.
2. Factors that are products themselves are replaced by their factors.  As $\varepsilon$ is the empty product,
   it vanishes.
3. If $y$ is a nullable factor next to a factor $x^*$ and $L(y) \subseteq L(x^*)$, then $y$ is dropped, since
   $$ L(x^*) \subseteq L(x^* \cdot y) \subseteq L(x^* \cdot x^*) = L(x^*). $$
   For example, $r^* \cdot (r + \varepsilon) \doteq r^*$ and $r^* \cdot r^* \doteq r^*$.
4. A product with only one factor is this factor.

**Function `mk_prod f`**
- *Input:* `f` is a list of flattened regular expressions.
- *Output:* A simplified flattened regular expression that is equivalent to the product of the elements of `f`.

The Python version deletes factors from a list while it walks through this list with an index `i`.  The OCaml
version uses two lists instead: `left` contains the factors that have already been checked (in reverse order) and
`right` contains the remaining factors.  If the factor `x` is dropped, `i` is decremented in Python.  Here, the
last checked factor is moved back from `left` to `right`.

In [ ]:
let is_star (r : flat) : bool =
  match r with FStar _ -> true | _ -> false

let mk_prod : flat list -> flat =
  cache (fun _ f ->
    if List.mem empty f then empty
    else
      let rec go (left : flat list) (right : flat list) : flat list =
        match right with
        | x :: y :: rest when is_star x && nullable y && included y x -> go left (x :: rest)
        | x :: (y :: _ as tl) when is_star y && nullable x && included x y -> (
            match left with
            | [] -> go [] tl
            | l :: ls -> go ls (l :: tl))
        | x :: tl -> go (x :: left) tl
        | [] -> List.rev left
      in
      match go [] (List.concat_map factors f) with
      | [ x ] -> x
      | fs -> FProd fs)
;;
mk_prod_ref := mk_prod

Inside a Kleene closure, many regular expressions can be replaced by simpler ones.  The function `strip r`
computes a regular expression $s$ such that $s^* \doteq r^*$ using the following laws:
1. $(r^*)^* \doteq r^*$.
2. $(\varepsilon + r)^* \doteq r^*$ and, more generally, $(r_1 + \cdots + r_n)^* \doteq (s_1 + \cdots + s_n)^*$ if
   $s_i^* \doteq r_i^*$ for all $i$, since $(r_1 + r_2)^* \doteq (r_1^* + r_2^*)^*$.  For example,
   $(r^* + s)^* \doteq (r + s)^*$.
3. If all factors $r_1$, $\cdots$, $r_n$ are nullable, then $(r_1 \cdot \ldots \cdot r_n)^* \doteq (r_1 + \cdots + r_n)^*$,
   since every $r_i$ is contained in the product.  For example, $(\texttt{a}^* \cdot \texttt{b}^*)^* \doteq (\texttt{a} + \texttt{b})^*$.

As $\varepsilon$ is the empty product, the third law turns $\varepsilon$ into the empty sum $\emptyset$.

**Function `strip r`**
- *Input:* `r` is a flattened regular expression.
- *Output:* A flattened regular expression $s$ such that $s^* \doteq r^*$.

In [ ]:
let rec strip (r : flat) : flat =
  match r with
  | FStar s -> strip s
  | FSum s -> mk_sum (FSet.map strip s)
  | FProd f when List.for_all nullable f -> mk_sum (FSet.of_list (List.map strip f))
  | _ -> r

The smart constructor `mk_star r` builds the Kleene closure of `r`.  Since $\emptyset^* \doteq \varepsilon$ and
$\varepsilon^* \doteq \varepsilon$, the result is $\varepsilon$ if `strip r` is $\emptyset$ or $\varepsilon$.

**Function `mk_star r`**
- *Input:* `r` is a flattened regular expression.
- *Output:* A simplified flattened regular expression that is equivalent to $r^*$.

In [ ]:
let mk_star : flat -> flat =
  cache (fun _ r ->
    let s = strip r in
    if s = empty || s = epsilon then epsilon else FStar s)

## Simplification

The function `flatten r` converts a regular expression `r` into the flattened representation.  It traverses
`r` bottom-up and builds the result with the smart constructors.  Hence the result is already simplified.

**Function `flatten r`**
- *Input:* `r` is a regular expression, represented as a value of type `regexp`.
- *Output:* A simplified flattened regular expression that is equivalent to `r`.

In [ ]:
let rec flatten (r : regexp) : flat =
  match r with
  | Empty -> empty
  | Epsilon -> epsilon
  | Char c -> FChar c
  | Star r1 -> mk_star (flatten r1)
  | Sum (r1, r2) -> mk_sum (FSet.of_list [ flatten r1; flatten r2 ])
  | Product (r1, r2) -> mk_prod [ flatten r1; flatten r2 ]

The function `unflatten r` converts a flattened regular expression back into a value of type `regexp`.  Sums and
products become left-nested trees, as they are produced by the function `parse`.  The summands of a sum are
sorted by their keys and $\varepsilon$ is put at the end, so that the result does not depend on the order of the
elements of a set.

**Function `unflatten r`**
- *Input:* `r` is a flattened regular expression.
- *Output:* The regular expression represented by `r` as a value of type `regexp`.

In [ ]:
let rec unflatten (r : flat) : regexp =
  match r with
  | FStar s -> Star (unflatten s)
  | FSum s when FSet.is_empty s -> Empty
  | FSum s ->
      let rs = List.map unflatten (sorted_by_key (FSet.to_list (FSet.remove epsilon s))) in
      let rs = if FSet.mem epsilon s then rs @ [ Epsilon ] else rs in
      List.fold_left (fun result s -> Sum (result, s)) (List.hd rs) (List.tl rs)
  | FProd [] -> Epsilon
  | FProd (f0 :: fs) -> List.fold_left (fun result f -> Product (result, unflatten f)) (unflatten f0) fs
  | FChar c -> Char c
;;
unflatten_ref := unflatten

**Function `simplify r`**
- *Input:* `r` is a regular expression, represented as a value of type `regexp`.
- *Output:* A simplified regular expression that is equivalent to `r`.

In [ ]:
let simplify (r : regexp) : regexp =
  unflatten (flatten r)